# Build the `pool` split by concatenating `medium` + `hard`

For **every** model in `nmscrc.MODELS`, concatenate (medium first, hard after) and write the
pooled files back to the same raw-data folder:

| inputs (medium, then hard) | output |
| --- | --- |
| `<model>_medium_data.csv`, `<model>_hard_data.csv` | `<model>_pool_data.csv` |
| `<model>_medium_hs.npy`, `<model>_hard_hs.npy` | `<model>_pool_hs.npy` |

The CSV is concatenated row-wise (index reset) and the hidden-state array along axis 0, so row *i*
of the pooled CSV stays aligned with row *i* of the pooled `.npy`. Paths come from `nmscrc.paths`
(env-overridable; no absolute path hard-coded). The raw-data folder is `paths._raw_dir()`
(`NMSCRC_RAW_DIR` env > `config.yaml: raw_dir` > `data-full`).

In [ ]:
# Resolve paths via the package — no hard-coded absolute path. Works whether or not `nmscrc`
# is pip-installed: walk up to the repo root (the dir holding config.yaml) and add it to sys.path.
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "config.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from nmscrc import paths

DATA_DIR = paths.DATA_ROOT / paths._raw_dir()   # raw-data folder for this run
MODELS = paths.MODELS
print(f"DATA_DIR = {DATA_DIR}")
print(f"MODELS   = {MODELS}")

def p(model_name, split, kind, ext):
    return DATA_DIR / f"{model_name}_{split}_{kind}.{ext}"


In [ ]:
def build_pool(model_name):
    """Concatenate medium+hard -> pool for one model, keeping CSV and hs row-aligned."""
    # --- CSV: medium then hard ---
    medium_df = pd.read_csv(p(model_name, "medium", "data", "csv"))
    hard_df = pd.read_csv(p(model_name, "hard", "data", "csv"))
    assert list(medium_df.columns) == list(hard_df.columns), f"{model_name}: column mismatch"
    pool_df = pd.concat([medium_df, hard_df], axis=0, ignore_index=True)
    pool_df.to_csv(p(model_name, "pool", "data", "csv"), index=False)

    # --- hidden states: medium then hard ---
    medium_hs = np.load(p(model_name, "medium", "hs", "npy"))
    hard_hs = np.load(p(model_name, "hard", "hs", "npy"))
    assert medium_hs.shape[1:] == hard_hs.shape[1:], f"{model_name}: feature-dim mismatch"
    pool_hs = np.concatenate([medium_hs, hard_hs], axis=0)
    np.save(p(model_name, "pool", "hs", "npy"), pool_hs)

    # --- sanity: pooled CSV rows must equal pooled hs rows (kept in lockstep) ---
    assert len(pool_df) == pool_hs.shape[0], f"{model_name}: row count mismatch CSV vs hs"
    print(f"[+] {model_name}: {medium_df.shape[0]} (medium) + {hard_df.shape[0]} (hard) "
          f"-> {len(pool_df)} rows aligned across pool data.csv / hs.npy")


In [ ]:
for m in MODELS:
    build_pool(m)
print("done.")
